# Build the Lakehouse tables + semantic model

Reads the committed synthetic FOCUS dataset (`focus_sample.parquet`) that you
uploaded to a Lakehouse, then builds everything the ACA app + Data Agent need:

- **Silver**: `focus_partitioned` (daily FOCUS line items, partitioned by Year/Month)
- **Gold** (identical schema/logic to the production ACA accelerator notebooks 02-08):
  `dim_date`, `dim_month`, `gold_cost_summary_daily`, `gold_cost_summary_monthly`,
  `gold_cost_focus_monthly`, `gold_chargeback_by_tag` (WIDE dynamic tags) + `dim_tag_key`,
  `gold_reservations_coverage`, `gold_reservations_waste`, `gold_reservations_detail`,
  `gold_cost_anomalies`, `gold_cost_by_resource` (tag-free — keeps only `TagCount`)
- A **Direct Lake (on SQL endpoint)** semantic model binding the gold tables (via `sempy_labs`)

The tables match production exactly, so the accelerator's semantic model / report /
app work unchanged against this Lakehouse — **only the data differs**.

**Manual prerequisites (one time — see README.md):**
1. Create a **schema-enabled** Lakehouse in your workspace (default name: `AzureCostAnalyzer_LH`).
2. Upload `app/sample-data/focus_sample.parquet` into that Lakehouse's `Files/`.
3. Import THIS file as a Notebook and **attach the Lakehouse as its default**.
4. Set the parameters below, then **Run all**.


In [ ]:
# Install semantic-link-labs FIRST. In Fabric, %pip can restart the Python session,
# which clears variables defined in earlier cells. Keeping it as the first cell (before
# the parameters) ensures everything after it runs in the post-restart session.
%pip install semantic-link-labs --quiet

In [ ]:
lakehouse_name = "AzureCostAnalyzer_LH"    # Lakehouse you created + uploaded the parquet to
parquet_file   = "focus_sample.parquet"    # file uploaded under <lakehouse>/Files/
model_name     = "AzureCostAnalyzer_SM"    # semantic model to (re)create

## `focus_partitioned` — silver (partitioned by Year/Month)

Mirrors production notebook `01_Load_CostManagement_Focus_Data`: the parquet already
is the FOCUS silver schema, so we persist it as a partitioned Delta table, then derive
a working frame with the `Date` / `YearMonth` the gold layer groups on.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

focus_raw = spark.read.parquet(f"Files/{parquet_file}")
# Parquet timestamps read back as TIMESTAMP_NTZ (no time zone); writing that to Delta
# would require manually enabling the `timestampNtz` table feature (raises min reader/
# writer versions and can break the SQL endpoint / Direct Lake). Cast to standard
# TIMESTAMP instead — matches the production focus_partitioned schema.
_TS_COLS = ["ChargePeriodStart", "ChargePeriodEnd", "BillingPeriodStart", "BillingPeriodEnd"]
for _c in _TS_COLS:
    if _c in focus_raw.columns:
        focus_raw = focus_raw.withColumn(_c, F.col(_c).cast("timestamp"))

# Persist Date + YearMonth + a friendly ResourceGroupName ON the table so the semantic
# model can bind focus_partitioned (relationship to dim_date[Date] + the FX measures +
# Explorer "what changed" / anomaly window queries reference these columns).
focus_raw = (focus_raw
    .withColumn("Date", F.to_date("ChargePeriodStart"))
    .withColumn("YearMonth", F.date_format("ChargePeriodStart", "yyyy-MM"))
    .withColumn("ResourceGroupName", F.col("x_ResourceGroupName")))

(focus_raw.write.mode("overwrite").option("overwriteSchema", "true")
          .partitionBy("Year", "Month").format("delta").saveAsTable("focus_partitioned"))

f = spark.table("focus_partitioned")
usage = f.filter(F.col("ChargeCategory") == "Usage")
print(f"focus_partitioned rows: {f.count():,}  |  months: {f.select('YearMonth').distinct().count()}")


## `dim_date` / `dim_month` (production notebook 02)

In [ ]:
_b = f.select(F.min("Date").alias("mn"), F.max("Date").alias("mx")).first()
dim_date = (spark.sql(f"SELECT explode(sequence(DATE'{_b.mn}', DATE'{_b.mx}', INTERVAL 1 DAY)) AS Date")
    .withColumn("Year", F.year("Date"))
    .withColumn("Month", F.month("Date"))
    .withColumn("YearMonth", F.date_format("Date", "yyyy-MM"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("QuarterLabel", F.concat(F.lit("Q"), F.quarter("Date"), F.lit(" "), F.year("Date")))
    .withColumn("WeekOfYear", F.weekofyear("Date"))
    .withColumn("YearWeek", F.concat(F.year("Date"), F.lit("-W"), F.lpad(F.weekofyear("Date"), 2, "0")))
    .withColumn("DayOfMonth", F.dayofmonth("Date"))
    .withColumn("DayOfWeek", F.dayofweek("Date"))
    .withColumn("DayName", F.date_format("Date", "EEEE"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("DateKey", F.date_format("Date", "yyyyMMdd").cast("int"))
    .withColumn("IsPast", F.col("Date") <= F.current_date())
    .withColumn("IsCurrentMonth", F.date_format("Date", "yyyy-MM") == F.date_format(F.current_date(), "yyyy-MM")))
dim_date.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("dim_date")

dim_month = (dim_date.groupBy("YearMonth")
    .agg(F.min("Date").alias("MonthStart"), F.first("Year").alias("Year"),
         F.first("Month").alias("MonthNum"), F.first("MonthName").alias("MonthName"))
    .withColumn("Value", F.lit(1)))
dim_month.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("dim_month")

## `gold_cost_summary_daily` / `gold_cost_summary_monthly` (production notebook 04)

Core cost facts (Effective / Billed / List / Savings). `ChargeCategory = 'Usage'` only.

In [ ]:
gold_cost_summary_daily = (usage.groupBy("Date", "SubAccountName", "ServiceCategory", "ServiceName", "RegionName")
    .agg(F.sum("EffectiveCost").alias("EffectiveCost"), F.sum("BilledCost").alias("BilledCost"),
         F.sum("ListCost").alias("ListCost"),
         F.sum(F.col("ListCost") - F.col("EffectiveCost")).alias("SavingsAmount")))
gold_cost_summary_daily.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_cost_summary_daily")

gold_cost_summary_monthly = (usage.groupBy("YearMonth", "SubAccountName", "ServiceCategory", "ServiceName")
    .agg(F.sum("EffectiveCost").alias("EffectiveCost"), F.sum("BilledCost").alias("BilledCost"),
         F.sum("ListCost").alias("ListCost"),
         F.sum(F.col("ListCost") - F.col("EffectiveCost")).alias("SavingsAmount")))
gold_cost_summary_monthly.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_cost_summary_monthly")

## `gold_cost_focus_monthly` (production notebook 03)

Wide FOCUS fact with the **hybrid savings baseline** (Committed rows use `ContractedCost`,
everything else uses `ListCost`) — fixes amortized-RI rows whose `ListCost = 0`.

In [ ]:
_baseline = F.when(F.col("PricingCategory") == "Committed", F.col("ContractedCost")).otherwise(F.col("ListCost"))
gold_cost_focus_monthly = (f.withColumn("_BaselineRow", _baseline).groupBy(
        "YearMonth", "SubAccountName",
        F.col("x_ResourceGroupName").alias("ResourceGroupName"),
        "ServiceCategory", "ServiceName", "RegionName", "ChargeCategory", "PricingCategory",
        F.coalesce(F.col("CommitmentDiscountStatus"), F.lit("N/A")).alias("CommitmentDiscountStatus"),
        F.col("x_CostCenter").alias("CostCenter"))
    .agg(F.sum("EffectiveCost").alias("EffectiveCost"), F.sum("BilledCost").alias("BilledCost"),
         F.sum("ListCost").alias("ListCost"), F.sum("ContractedCost").alias("ContractedCost"),
         F.sum("_BaselineRow").alias("SavingsBaseline"),
         (F.sum("_BaselineRow") - F.sum("EffectiveCost")).alias("SavingsAmount"),
         F.sum("x_EffectiveCostInUsd").alias("EffectiveCostUSD"),
         F.countDistinct("ResourceId").alias("ResourceCount")))
gold_cost_focus_monthly.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_cost_focus_monthly")

## `gold_chargeback_by_tag` (WIDE, dynamic) + `dim_tag_key`

The single source for every **tag-driven** view (Chargeback multi-tag grouping + Action Center
governance & inline tagging).

- **Dynamic WIDE** — tag keys are discovered from the FOCUS `Tags` JSON (no hardcoded dimensions),
  normalized `trim`+`lower`; **every** key becomes its own column (its value, or `"Untagged"` when
  the resource lacks that key). Not just the top-N — all of them.
- **Resource grain** — one row per `(resource, month)`, so costs are counted **exactly once**. You can
  **group/filter by several tags at the same time** with no double counting.
- **`TagCount`** — number of tags on the resource (`0` = fully untagged) → governance / Action Center.
- **`dim_tag_key`** — maps each discovered key → its column (`ColumnName`) + display + rank, so the app
  discovers the (dynamic) tag columns at runtime and renders friendly names / builds its GROUP BY.
- Column names are sanitized to safe **PascalCase** (`cost center` → `CostCenter`); `dim_tag_key` keeps
  the original key + a friendly display, so odd characters never break a query.


In [ ]:
# gold_chargeback_by_tag — RESOURCE-GRAIN, DYNAMIC *WIDE* tag table (one column per tag key).
# The single source for every tag-driven view (Chargeback multi-tag grouping + Action Center
# governance / Tag-now). Tag KEYS are discovered from the FOCUS `Tags` JSON (NO hardcoded
# dimensions), normalized (trim+lower) so casing variants collapse. EVERY discovered key becomes
# its own column (its value, or "Untagged" when the resource lacks that key). ONE row per
# resource-month => costs counted EXACTLY ONCE, so you can group/filter by SEVERAL tags at a time
# with NO double counting. `dim_tag_key` maps each key -> its column so the app discovers the
# (dynamic) tag columns at runtime.
import re

UNTAGGED = "Untagged"

# Lower-casing keys can collapse case-variants onto the SAME key. Make the last value win instead
# of raising "Duplicate map key" when building the tag map.
spark.conf.set("spark.sql.mapKeyDedupPolicy", "LAST_WIN")

def _dflt(col, default):
    c = F.trim(col)
    return F.when(c.isNull() | (c == ""), F.lit(default)).otherwise(c)

# 1) Normalize Tags JSON -> map<key,value> (keys trim+lower, values trim, empties dropped).
_norm = usage.withColumn(
    "tagmap",
    F.map_from_entries(
        F.filter(
            F.transform(
                F.map_entries(F.from_json("Tags", "map<string,string>")),
                lambda e: F.struct(F.trim(F.lower(e["key"])).alias("key"), F.trim(e["value"]).alias("value")),
            ),
            lambda e: e["key"].isNotNull() & (e["key"] != "") & e["value"].isNotNull() & (e["value"] != ""),
        )
    ),
)

# 2) Discover EVERY tag key (ranked by cost) — this drives the dynamic columns.
_key_cost = (_norm.select("EffectiveCost", F.explode("tagmap").alias("k", "v"))
    .groupBy("k").agg(F.sum("EffectiveCost").alias("EffectiveCost"))
    .orderBy(F.desc("EffectiveCost")))
_keys = [(r["k"], float(r["EffectiveCost"])) for r in _key_cost.collect()]

# 3) Safe, readable PascalCase column name per key + friendly display. Dedupe CASE-INSENSITIVELY —
#    Spark resolves columns case-insensitively by default, so `Databricksenvironment` and
#    `DatabricksEnvironment` are the SAME column (would raise COLUMN_ALREADY_EXISTS). Also avoid
#    colliding with the fixed (non-tag) columns. Collisions get a numeric suffix.
def _colname(key):
    parts = [p for p in re.split(r"[^0-9a-z]+", key) if p]
    return "".join(p.capitalize() for p in parts) or "Tag"

_FIXED = ["YearMonth", "SubAccountName", "ServiceCategory", "ServiceName", "RegionName",
          "ResourceGroupName", "ResourceId", "ResourceName", "ResourceType",
          "EffectiveCost", "BilledCost", "TagCount"]
_used = {c.lower() for c in _FIXED}
TAG_MAP = []   # rows: (rank, original_key, column_name, display, cost)
for i, (k, c) in enumerate(_keys, start=1):
    base = _colname(k)
    col, n = base, 1
    while col.lower() in _used:
        n += 1
        col = f"{base}{n}"
    _used.add(col.lower())
    TAG_MAP.append((i, k, col, k.title(), c))

# 4) Build the WIDE resource-grain fact: one column per key (value or "Untagged") + TagCount.
_res_type = _dflt(F.regexp_extract(F.lower(F.col("ResourceId")), r"providers/([^/]+/[^/]+)", 1), "(unknown)")
_r = _norm.select(
    "YearMonth", "SubAccountName", "ServiceCategory", "ServiceName", "RegionName",
    _dflt(F.col("x_ResourceGroupName"), "(no resource group)").alias("ResourceGroupName"),
    _dflt(F.col("ResourceId"), "(no resource)").alias("ResourceId"),
    _dflt(F.col("ResourceName"), "(no resource)").alias("ResourceName"),
    _res_type.alias("ResourceType"),
    "EffectiveCost", "BilledCost",
    F.when(F.col("tagmap").isNull(), F.lit(0)).otherwise(F.size("tagmap")).alias("TagCount"),
    F.col("tagmap"),
)
for (_, k, col, _, _) in TAG_MAP:
    _r = _r.withColumn(col, F.coalesce(_r["tagmap"].getItem(k), F.lit(UNTAGGED)))

_tag_cols = [col for (_, _, col, _, _) in TAG_MAP]
_group = ["YearMonth", "SubAccountName", "ServiceCategory", "ServiceName", "RegionName",
          "ResourceGroupName", "ResourceId", "ResourceName", "ResourceType"] + _tag_cols
gold_chargeback_by_tag = (_r.groupBy(*_group)
    .agg(F.sum("EffectiveCost").alias("EffectiveCost"),
         F.sum("BilledCost").alias("BilledCost"),
         F.max("TagCount").alias("TagCount")))
gold_chargeback_by_tag.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_chargeback_by_tag")

# 5) dim_tag_key — maps each discovered key -> its (dynamic) column + display + rank; the app reads
#    this to know which tag columns exist and to render friendly names / build its GROUP BY.
dim_tag_key = spark.createDataFrame(
    [(rank, key, col, disp, cost) for (rank, key, col, disp, cost) in TAG_MAP],
    "Rank int, TagKey string, ColumnName string, TagKeyDisplay string, EffectiveCost double",
).select("TagKey", "TagKeyDisplay", "ColumnName", "Rank", "EffectiveCost")
dim_tag_key.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("dim_tag_key")

print(f"✓ gold_chargeback_by_tag (WIDE, dynamic): {gold_chargeback_by_tag.count():,} rows, {len(_tag_cols)} tag columns")
print(f"  tag columns: {_tag_cols}")
print(f"✓ dim_tag_key (customer tag universe): {len(TAG_MAP)} keys")
dim_tag_key.orderBy("Rank").show(50, truncate=False)


## `gold_reservations_*` — coverage / waste / detail (production notebook 06)

FOCUS 1.0 commitment semantics: committed usage = `PricingCategory = 'Committed'`;
unused commitment = `CommitmentDiscountStatus = 'Unused'`.

In [ ]:
gold_reservations_coverage = (usage.groupBy("YearMonth", "SubAccountName")
    .agg(F.sum(F.when(F.col("PricingCategory") == "Committed", F.col("EffectiveCost")).otherwise(F.lit(0.0))).alias("ReservedCost"),
         F.sum("EffectiveCost").alias("TotalCost"))
    .withColumn("CoveragePercent",
        F.when(F.col("TotalCost") != 0, F.col("ReservedCost") / F.col("TotalCost") * 100).otherwise(F.lit(None)).cast("double")))
gold_reservations_coverage.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_reservations_coverage")

_res = usage.filter(F.col("CommitmentDiscountType").isNotNull())
gold_reservations_waste = (_res.groupBy("YearMonth", "SubAccountName")
    .agg(F.sum(F.when(F.col("CommitmentDiscountStatus") == "Unused", F.col("EffectiveCost")).otherwise(F.lit(0.0))).cast("double").alias("WasteCost"),
         F.sum(F.when(F.col("CommitmentDiscountStatus") == "Used", F.col("EffectiveCost")).otherwise(F.lit(0.0))).alias("_used"),
         F.sum(F.when(F.col("CommitmentDiscountStatus") == "Unused", F.col("EffectiveCost")).otherwise(F.lit(0.0))).alias("_unused"))
    .withColumn("UtilizationPercent",
        F.when((F.col("_used") + F.col("_unused")) > 0, F.col("_used") / (F.col("_used") + F.col("_unused")) * 100).otherwise(F.lit(None)).cast("double"))
    .select("YearMonth", "SubAccountName", "WasteCost", "UtilizationPercent"))
gold_reservations_waste.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_reservations_waste")

_det = usage.filter((F.col("PricingCategory") == "Committed") & F.col("CommitmentDiscountType").isNotNull())
gold_reservations_detail = (_det.groupBy("YearMonth", "SubAccountName", "CommitmentDiscountId",
        F.coalesce(F.col("CommitmentDiscountName"), F.lit("(sin nombre)")).alias("CommitmentDiscountName"),
        F.coalesce(F.col("CommitmentDiscountType"), F.lit("(desconocido)")).alias("CommitmentDiscountType"),
        F.coalesce(F.col("CommitmentDiscountCategory"), F.lit("(desconocido)")).alias("CommitmentDiscountCategory"),
        "ServiceName", "RegionName")
    .agg(F.sum(F.when(F.col("CommitmentDiscountStatus") == "Used", F.col("EffectiveCost")).otherwise(F.lit(0.0))).alias("UsedCost"),
         F.sum(F.when(F.col("CommitmentDiscountStatus") == "Unused", F.col("EffectiveCost")).otherwise(F.lit(0.0))).alias("WasteCost"),
         F.sum(F.when(F.col("CommitmentDiscountStatus") == "Used", F.col("ContractedCost")).otherwise(F.lit(0.0))).alias("ContractedCostUsed"))
    .withColumn("TotalCommitmentCost", F.col("UsedCost") + F.col("WasteCost"))
    .withColumn("SavingsUsed", F.col("ContractedCostUsed") - F.col("UsedCost"))
    .withColumn("UtilizationPercent",
        F.when((F.col("UsedCost") + F.col("WasteCost")) > 0, F.col("UsedCost") / (F.col("UsedCost") + F.col("WasteCost")) * 100).otherwise(F.lit(None))))
gold_reservations_detail.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_reservations_detail")

## `gold_cost_anomalies` (production notebook 07)

7-day rolling mean/std-dev per (SubAccountName, ServiceName); Z-score > 3 = anomaly.

In [ ]:
_anom = (spark.table("gold_cost_summary_daily")
    .groupBy("Date", "SubAccountName", "ServiceCategory", "ServiceName")
    .agg(F.sum("EffectiveCost").alias("EffectiveCost"), F.first("RegionName").alias("RegionName")))
_w7 = Window.partitionBy("SubAccountName", "ServiceName").orderBy("Date").rowsBetween(-6, 0)
gold_cost_anomalies = (_anom
    .withColumn("RollingMean", F.avg("EffectiveCost").over(_w7))
    .withColumn("RollingStdDev", F.stddev_samp("EffectiveCost").over(_w7))
    .withColumn("ZScore", F.when(F.col("RollingStdDev") > 0,
        (F.col("EffectiveCost") - F.col("RollingMean")) / F.col("RollingStdDev")).otherwise(F.lit(0.0)))
    .withColumn("IsAnomaly", F.abs(F.col("ZScore")) > 3)
    .withColumn("Severity", F.when(F.abs(F.col("ZScore")) > 5, F.lit("Critical"))
                             .when(F.abs(F.col("ZScore")) > 3, F.lit("High")).otherwise(F.lit("Normal")))
    .select("Date", "SubAccountName", "ServiceCategory", "ServiceName", "RegionName", "EffectiveCost",
            "RollingMean", "RollingStdDev", "ZScore", "IsAnomaly", "Severity"))
gold_cost_anomalies.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_cost_anomalies")

## `gold_cost_by_resource` (production notebook 08)

Per-resource monthly cost for **Explorer / Resource Detail / Governance**. **Tags no longer live
here** — the dynamic tag model is `gold_chargeback_by_tag` (WIDE) + `dim_tag_key`. This table keeps
only **`TagCount`** (`0` = fully untagged) so the Governance measures work without any tag columns,
and stays lean for the non-tag views.


In [ ]:
def _clean(expr, default):
    c = F.trim(expr)
    return F.when(c.isNull() | (c == ""), F.lit(default)).otherwise(c)

# Per-resource monthly cost for Explorer / Resource Detail / Governance.
# TAGS NO LONGER LIVE HERE — the dynamic tag model is `gold_chargeback_by_tag` (WIDE) + `dim_tag_key`.
# We keep only `TagCount` (0 = fully untagged) so the Governance measures work with NO tag columns,
# and this table stays lean for the non-tag views.
_rt = usage.withColumn("tagmap", F.from_json("Tags", "map<string,string>"))
_r2 = (_rt
    .withColumn("_TagMapSize", F.when(F.col("tagmap").isNull(), F.lit(0)).otherwise(F.size("tagmap")))
    .withColumn("_BaselineRow", F.when(F.col("PricingCategory") == "Committed", F.col("ContractedCost")).otherwise(F.col("ListCost")))
    .withColumn("_ResourceType",
        _clean(F.regexp_extract(F.lower(F.col("ResourceId")), r"providers/([^/]+/[^/]+)", 1), "(unknown)")))

_group = ["YearMonth", "SubAccountName",
    _clean(F.col("x_ResourceGroupName"), "(no resource group)").alias("ResourceGroupName"),
    _clean(F.col("ResourceId"), "(no resource)").alias("ResourceId"),
    _clean(F.col("ResourceName"), "(no resource)").alias("ResourceName"),
    F.col("_ResourceType").alias("ResourceType"),
    "ServiceCategory", "ServiceName", "RegionName"]
gold_cost_by_resource = (_r2.groupBy(*_group)
    .agg(F.sum("EffectiveCost").alias("EffectiveCost"), F.sum("BilledCost").alias("BilledCost"),
         F.sum("ListCost").alias("ListCost"),
         (F.sum("_BaselineRow") - F.sum("EffectiveCost")).alias("SavingsAmount"),
         F.max("_TagMapSize").alias("TagCount")))
gold_cost_by_resource.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_cost_by_resource")

print("Delta tables written to", lakehouse_name)
print("gold_cost_by_resource: tags removed — sourced from gold_chargeback_by_tag / dim_tag_key; TagCount kept for governance.")


## Semantic model — create Direct Lake (on SQL endpoint)

`semantic-link-labs` (`sempy_labs`) is the automation library. API can vary by
version; pin/upgrade if a signature differs.

In [ ]:
import sempy_labs as labs
import sempy.fabric as fabric
from sempy_labs.tom import connect_semantic_model

# Schema-enabled Lakehouse: Spark writes tables under Tables/dbo/. sempy needs the
# source schema-qualified. Dict form keeps clean model table names (key) mapped to
# the schema-qualified lakehouse table (value = "dbo.<table>").
# Bind focus_partitioned (silver — powers the FX measures + Explorer/anomaly deep-dives)
# plus the gold tables. `gold_chargeback_by_tag` is the WIDE dynamic tag table; `dim_tag_key`
# is the tag-universe helper the app reads to discover the (dynamic) tag columns — bound with
# NO relationships (it's a lookup the app EVALUATEs, not a filter dimension).
MODEL_TABLES = ["focus_partitioned", "gold_cost_summary_daily", "gold_cost_summary_monthly",
                "gold_cost_focus_monthly", "gold_chargeback_by_tag", "dim_tag_key", "gold_cost_by_resource",
                "gold_reservations_coverage", "gold_reservations_waste", "gold_reservations_detail",
                "gold_cost_anomalies", "dim_date", "dim_month"]
TABLES = {t: f"dbo.{t}" for t in MODEL_TABLES}

# Direct-Lake-on-SQL reads through the Lakehouse SQL analytics endpoint. The tables were
# just (re)written by Spark, so the endpoint metadata can lag — force + wait for the sync.
print("Syncing SQL endpoint metadata (this can take a minute)...")
display(labs.refresh_sql_endpoint_metadata(item=lakehouse_name, type="Lakehouse"))

def _model_exists(name):
    try:
        df = fabric.list_datasets()
        col = next((c for c in df.columns if c.lower() in ("dataset name", "name")), df.columns[0])
        return name in set(df[col].astype(str))
    except Exception:
        return False

# Reuse the existing model if present (keeps the SAME itemId, so the app's fabric.yaml
# stays valid across re-runs). Only create a NEW model when it doesn't exist yet.
if _model_exists(model_name):
    print(f"Reusing existing model '{model_name}' (same itemId).")
    # Pick up any new/changed columns in the underlying Delta tables (e.g. the dynamic tag columns).
    try:
        labs.directlake.direct_lake_schema_sync(dataset=model_name, add_to_model=True)
        print("Schema synced (added any new lakehouse columns to the model).")
    except Exception as e:
        print(f"  schema sync skipped: {e}")
else:
    print(f"Creating new model '{model_name}'.")
    labs.directlake.generate_direct_lake_semantic_model(
        dataset=model_name,
        tables=TABLES,
        source=lakehouse_name,
        source_type="Lakehouse",
        use_sql_endpoint=True,
        refresh=False,
    )


## Relationships + date table + measures

In [ ]:
CUR = "\\$#,##0"        # currency format string
CUR2 = "\\$#,##0.00"
PCT = "0.0%"
NUM = "#,##0"

with connect_semantic_model(dataset=model_name, readonly=False) as tom:

    # Snapshot existing relationships so re-runs on a REUSED model don't add duplicates.
    # (add_relationship does NOT error on a duplicate — it silently adds a second one,
    # which then fails at save time with "ambiguous paths between ...".)
    _existing_rels = {
        (r.FromTable.Name, r.FromColumn.Name, r.ToTable.Name, r.ToColumn.Name)
        for r in tom.model.Relationships
    }

    def rel(ft, fc, tt, tc):
        if (ft, fc, tt, tc) in _existing_rels:
            return
        tom.add_relationship(
            from_table=ft, from_column=fc, to_table=tt, to_column=tc,
            from_cardinality="Many", to_cardinality="One",
            cross_filtering_behavior="OneDirection", is_active=True,
        )
        _existing_rels.add((ft, fc, tt, tc))

    # Daily facts -> dim_date ; monthly facts -> dim_month ; dim_date -> dim_month
    rel("focus_partitioned",          "Date",      "dim_date",  "Date")
    rel("gold_cost_summary_daily",    "Date",      "dim_date",  "Date")
    rel("gold_cost_anomalies",        "Date",      "dim_date",  "Date")
    rel("gold_cost_summary_monthly",  "YearMonth", "dim_month", "YearMonth")
    rel("gold_cost_focus_monthly",    "YearMonth", "dim_month", "YearMonth")
    rel("gold_chargeback_by_tag",     "YearMonth", "dim_month", "YearMonth")
    rel("gold_cost_by_resource",      "YearMonth", "dim_month", "YearMonth")
    rel("gold_reservations_coverage", "YearMonth", "dim_month", "YearMonth")
    rel("gold_reservations_waste",    "YearMonth", "dim_month", "YearMonth")
    rel("gold_reservations_detail",   "YearMonth", "dim_month", "YearMonth")
    rel("dim_date",                   "YearMonth", "dim_month", "YearMonth")

    try:
        tom.mark_as_date_table(table_name="dim_date", column_name="Date")
    except Exception:
        pass

    def m(tbl, name, expr, fmt=None, folder=None):
        # idempotent: drop any existing measure of the same name before (re)adding,
        # so re-runs against a reused model pick up expression/format changes.
        try:
            t = tom.model.Tables[tbl]
            for ms in list(t.Measures):
                if ms.Name == name:
                    t.Measures.Remove(ms)
        except Exception:
            pass
        tom.add_measure(table_name=tbl, measure_name=name, expression=expr,
                        format_string=fmt, display_folder=folder)

    # -- Cost --
    m("gold_cost_summary_monthly", "Total Effective Cost", "SUM('gold_cost_summary_monthly'[EffectiveCost])", CUR, "Cost")
    m("gold_cost_summary_monthly", "Total Billed Cost",    "SUM('gold_cost_summary_monthly'[BilledCost])",    CUR, "Cost")
    m("gold_cost_summary_monthly", "Total List Cost",      "SUM('gold_cost_summary_monthly'[ListCost])",      CUR, "Cost")
    m("gold_cost_summary_monthly", "Total Savings",        "SUM('gold_cost_summary_monthly'[SavingsAmount])", CUR, "Cost")
    m("gold_cost_summary_monthly", "Savings %",            "DIVIDE([Total Savings],[Total List Cost])",       PCT, "Cost")
    m("gold_cost_focus_monthly",   "Real Savings",         "SUM('gold_cost_focus_monthly'[SavingsAmount])",   CUR, "Cost")
    m("gold_cost_focus_monthly",   "Total Baseline Cost",  "SUM('gold_cost_focus_monthly'[SavingsBaseline])", CUR, "Cost")

    # -- FOCUS usage-vs-rate (home: focus_partitioned) — powers Explorer bridge + anomaly window --
    m("focus_partitioned", "FX Effective Cost",    "SUM('focus_partitioned'[EffectiveCost])",    CUR2, "FOCUS")
    m("focus_partitioned", "FX Consumed Quantity", "SUM('focus_partitioned'[ConsumedQuantity])", "#,##0.0", "FOCUS")

    # -- Time Intelligence (monthly facts relate via dim_month; use MonthStart for MoM) --
    m("gold_cost_summary_monthly", "Last Month Cost",
      "VAR d = MAX('dim_month'[MonthStart]) "
      "RETURN CALCULATE([Total Effective Cost], REMOVEFILTERS('dim_month'), 'dim_month'[MonthStart] = EDATE(d, -1))",
      CUR, "Time Intelligence")
    m("gold_cost_summary_monthly", "MoM Cost %",
      "DIVIDE([Total Effective Cost] - [Last Month Cost], [Last Month Cost])", PCT, "Time Intelligence")

    # -- Anomalies --
    m("gold_cost_anomalies", "Anomaly Count",
      "CALCULATE(COUNTROWS('gold_cost_anomalies'), 'gold_cost_anomalies'[IsAnomaly] = TRUE())", NUM, "Anomalies")
    m("gold_cost_anomalies", "Anomaly Count (KPI)", "[Anomaly Count]", NUM, "Anomalies")

    # -- Reservations --
    m("gold_reservations_coverage", "RI Coverage %",
      "DIVIDE(SUM('gold_reservations_coverage'[ReservedCost]), SUM('gold_reservations_coverage'[TotalCost]))", PCT, "Reservations")
    m("gold_reservations_waste",    "RI Waste Cost", "SUM('gold_reservations_waste'[WasteCost])", CUR, "Reservations")
    m("gold_cost_focus_monthly",    "Net Reservation Savings",
      "CALCULATE(SUM('gold_cost_focus_monthly'[SavingsAmount]), 'gold_cost_focus_monthly'[PricingCategory] = \"Committed\")", CUR, "Reservations")
    m("gold_reservations_waste",    "Used Reservation Savings", "SUM('gold_reservations_detail'[SavingsUsed])", CUR, "Reservations")
    m("gold_reservations_detail",   "Used Reservation Cost",    "SUM('gold_reservations_detail'[UsedCost])",   CUR, "Reservations")
    m("gold_reservations_detail",   "Wasted Reservation Cost",  "SUM('gold_reservations_detail'[WasteCost])",  CUR, "Reservations")
    m("gold_reservations_detail",   "Reservation Utilization %",
      "DIVIDE(SUM('gold_reservations_detail'[UsedCost]), SUM('gold_reservations_detail'[UsedCost]) + SUM('gold_reservations_detail'[WasteCost]))", PCT, "Reservations")

    # -- Governance (home: gold_cost_by_resource; TagCount = 0 means fully untagged) --
    m("gold_cost_by_resource", "Untagged Cost",
      "CALCULATE(SUM('gold_cost_by_resource'[EffectiveCost]), 'gold_cost_by_resource'[TagCount] = 0)", CUR, "Governance")
    m("gold_cost_by_resource", "Untagged %",
      "DIVIDE([Untagged Cost], SUM('gold_cost_by_resource'[EffectiveCost]))", PCT, "Governance")
    m("gold_cost_by_resource", "Untagged Resource Count",
      "CALCULATE(DISTINCTCOUNT('gold_cost_by_resource'[ResourceId]), 'gold_cost_by_resource'[TagCount] = 0)", NUM, "Governance")
    m("gold_cost_by_resource", "Tag Count", "SUM('gold_cost_by_resource'[TagCount])", NUM, "Governance")
    m("gold_cost_by_resource", "Tags In Filter",
      "CALCULATE(DISTINCTCOUNT('gold_cost_by_resource'[ResourceId]), 'gold_cost_by_resource'[TagCount] > 0)", NUM, "Governance")

    # -- Resource Detail --
    m("gold_cost_by_resource", "Resource Effective Cost", "SUM('gold_cost_by_resource'[EffectiveCost])", CUR, "Resource Detail")
    m("gold_cost_by_resource", "Resource Count", "DISTINCTCOUNT('gold_cost_by_resource'[ResourceId])", NUM, "Resource Detail")

    # -- Tags (home: gold_chargeback_by_tag WIDE; grouped by any tag COLUMN discovered via dim_tag_key) --
    # Cost is exact at ONE row per resource-month, so multi-tag grouping/filtering does NOT double count.
    m("gold_chargeback_by_tag", "Tag Effective Cost", "SUM('gold_chargeback_by_tag'[EffectiveCost])", CUR, "Tags")
    m("gold_chargeback_by_tag", "Tagged Resource Count",
      "CALCULATE(DISTINCTCOUNT('gold_chargeback_by_tag'[ResourceId]), 'gold_chargeback_by_tag'[TagCount] > 0)", NUM, "Tags")
    m("gold_chargeback_by_tag", "Untagged Resource Count (Tags)",
      "CALCULATE(DISTINCTCOUNT('gold_chargeback_by_tag'[ResourceId]), 'gold_chargeback_by_tag'[TagCount] = 0)", NUM, "Tags")


## Refresh (reframe Direct Lake) + validate

In [ ]:
import sempy.fabric as fabric

# Reframe Direct Lake (pick up the freshly written Delta data).
labs.refresh_semantic_model(dataset=model_name)

# Definitive validation: run a DAX query against the new model. If this returns a value,
# the model + measures + relationships all work end-to-end.
print("Sanity check (DAX against the new model):")
try:
    display(fabric.evaluate_dax(
        dataset=model_name,
        dax_string='EVALUATE ROW("Total Effective Cost", [Total Effective Cost], "Untagged %", [Untagged %])',
    ))
except Exception as e:
    print(f"  sanity DAX skipped: {e}")

# Best-effort measure listing (some sempy/TOM versions raise on Direct Lake models).
print("Measures:")
try:
    display(fabric.list_measures(dataset=model_name))
except Exception as e:
    print(f"  list_measures skipped: {e}")

print(f"\nDone. Point the app at model '{model_name}' (set workspaceId + itemId in app/fabric.yaml).")


## (optional) App connection IDs

Prints the `workspaceId` + `itemId` of the new model, ready to paste into the app's
`app/fabric.yaml` under `semanticModels.aca` so the app queries this model.


In [ ]:
import sempy.fabric as fabric

ws_id = fabric.get_workspace_id()
try:
    item_id = fabric.resolve_item_id(item_name=model_name, type="SemanticModel")
except Exception:
    ds = fabric.list_datasets()
    name_col = next(c for c in ds.columns if c.lower() in ("dataset name", "name"))
    id_col   = next(c for c in ds.columns if c.lower() in ("dataset id", "id"))
    item_id = ds.loc[ds[name_col] == model_name, id_col].iloc[0]

print(f"aca workspaceId: {ws_id}")
print(f"aca itemId:      {item_id}")
print("\n--- paste into app/fabric.yaml (profiles.default.semanticModels.aca) ---")
print(f"        workspaceId: {ws_id}")
print(f"        itemId: {item_id}")
